In [1]:
import pandas as pd
import pickle

# Load train_dataset 
train_data = pd.read_csv('datasets/train_dataset.csv')

# Separate features and target
X = train_data.drop(['isFraud'], axis=1)
y = train_data['isFraud']

# Load the 10-fold indices
with open("10_fold_indices.pkl", "rb") as f:
    splits = pickle.load(f)

print(f"Loaded {len(splits)} folds.")

Loaded 10 folds.


In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
)
import os
import pickle

# -------------------- XGBoost Parameters --------------------
params = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "learning_rate": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42,
    "n_jobs": -1,
    "tree_method": "hist",
    "device": "cuda"
}

# -------------------- Setup --------------------
os.makedirs("models/xgboost", exist_ok=True)

auc_scores = []
accuracy_scores = []
precision_0, precision_1 = [], []
recall_0, recall_1 = [], []
f1_0, f1_1 = [], []

categorical_cols = X.select_dtypes(include="object").columns.tolist()

# -------------------- Cross-Validation --------------------
for fold, (train_idx, val_idx) in enumerate(splits, 1):
    print(f"\n Fold {fold}")

    X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Label Encoding
    for col in categorical_cols:
        le = LabelEncoder()
        combined = pd.concat([X_train[col], X_val[col]], axis=0)
        le.fit(combined)
        X_train[col] = le.transform(X_train[col])
        X_val[col] = le.transform(X_val[col])

    # Scale numeric features
    numeric_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
    scaler = StandardScaler()
    X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
    X_val[numeric_cols] = scaler.transform(X_val[numeric_cols])

    # Train XGBoost
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval = xgb.DMatrix(X_val, label=y_val)

    model = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=5000,
        evals=[(dval, "validation")],
        early_stopping_rounds=50,
        verbose_eval=50
    )

    # Predict and Evaluate
    y_val_prob = model.predict(dval, iteration_range=(0, model.best_iteration))
    y_val_pred = (y_val_prob > 0.5).astype(int)

    auc = roc_auc_score(y_val, y_val_prob)
    acc = accuracy_score(y_val, y_val_pred)
    prec = precision_score(y_val, y_val_pred, average=None, zero_division=0)
    rec = recall_score(y_val, y_val_pred, average=None)
    f1 = f1_score(y_val, y_val_pred, average=None)

    # Track metrics
    auc_scores.append(auc)
    accuracy_scores.append(acc)
    precision_0.append(prec[0])
    precision_1.append(prec[1])
    recall_0.append(rec[0])
    recall_1.append(rec[1])
    f1_0.append(f1[0])
    f1_1.append(f1[1])

    print(f" Fold {fold} AUC-ROC : {auc:.4f} | Best Iteration: {model.best_iteration}")
    print(f"   Accuracy     : {acc:.4f}")
    print(f"   Precision_0  : {prec[0]:.4f} | Precision_1 : {prec[1]:.4f}")
    print(f"   Recall_0     : {rec[0]:.4f} | Recall_1    : {rec[1]:.4f}")
    print(f"   F1 Score_0   : {f1[0]:.4f} | F1 Score_1  : {f1[1]:.4f}")

    # Save model
    model.save_model(f"models/xgboost/fold_{fold}.json")

# -------------------- Summary --------------------
print("\n Cross-Validation Summary (XGBoost + Label Encoding + Scaling)")
for i in range(len(auc_scores)):
    print(
        f"Fold {i+1}: AUC={auc_scores[i]:.4f} | Acc={accuracy_scores[i]:.4f} | "
        f"Prec_0={precision_0[i]:.4f} | Prec_1={precision_1[i]:.4f} | "
        f"Recall_0={recall_0[i]:.4f} | Recall_1={recall_1[i]:.4f} | "
        f"F1_0={f1_0[i]:.4f} | F1_1={f1_1[i]:.4f}"
    )

print("\n Final Averages:")
print(f"AUC-ROC     : {np.mean(auc_scores):.4f} ± {np.std(auc_scores):.4f}")
print(f"Accuracy    : {np.mean(accuracy_scores):.4f}")
print(f"Precision_0 : {np.mean(precision_0):.4f}")
print(f"Precision_1 : {np.mean(precision_1):.4f}")
print(f"Recall_0    : {np.mean(recall_0):.4f}")
print(f"Recall_1    : {np.mean(recall_1):.4f}")
print(f"F1 Score_0  : {np.mean(f1_0):.4f}")
print(f"F1 Score_1  : {np.mean(f1_1):.4f}")



 Fold 1
[0]	validation-auc:0.78784
[50]	validation-auc:0.88911
[100]	validation-auc:0.90702
[150]	validation-auc:0.91665
[200]	validation-auc:0.92321
[250]	validation-auc:0.92737
[300]	validation-auc:0.93189
[350]	validation-auc:0.93496
[400]	validation-auc:0.93755
[450]	validation-auc:0.93945
[500]	validation-auc:0.94145
[550]	validation-auc:0.94323
[600]	validation-auc:0.94505
[650]	validation-auc:0.94654
[700]	validation-auc:0.94805
[750]	validation-auc:0.94881
[800]	validation-auc:0.95016
[850]	validation-auc:0.95150
[900]	validation-auc:0.95256
[950]	validation-auc:0.95345
[1000]	validation-auc:0.95431
[1050]	validation-auc:0.95546
[1100]	validation-auc:0.95609
[1150]	validation-auc:0.95682
[1200]	validation-auc:0.95767
[1250]	validation-auc:0.95848
[1300]	validation-auc:0.95947
[1350]	validation-auc:0.96012
[1400]	validation-auc:0.96073
[1450]	validation-auc:0.96125
[1500]	validation-auc:0.96193
[1550]	validation-auc:0.96246
[1600]	validation-auc:0.96281
[1650]	validation-auc:0.